# PKG storage inventory (cdi_tm quota)

The database folder of `bdahd01p_dlcdi1_cdi_tm` has an HDFS space quota (9 TB on 2026-11-10). The quota counts
**space consumed**, which is the file size × the replication factor (usually 3), so 9 TB holds only about 3 TB of data.
Every table in the database shares it, including tables other people write.

1. Setup and settings.
2. **Inventory (read-only):** the quota, every folder under the database folder (one level deeper for state, work,
   temp and trash folders), its size, owner, last change, month partitions, the table that points to it, and what
   it is for (PKG notebooks only).
3. **Clean-up (does nothing by default):** drops the tables in `DROP_TABLES` and deletes the folders in
   `DELETE_PATHS`, skipping the trash so the quota frees at once. `DRY_RUN = True` only lists what would go.

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
DEEP = ("_", ".Trash")          # folders starting with these are listed one level deeper too
LOOK_INSIDE = ["pkg_cust_churn_state_v4", "_state_pkg_cust_reg", "_work_pkg_cust_reg"]   # §2b: every child + hidden space

# §3 clean-up list (2026-11-10). Run §3 with DRY_RUN = True first, check the list, then set DRY_RUN = False.
# Do not run it while a periodicity or customer-metrics build is running (their work and state folders are listed).
DELETE_PATHS = [
    "_work_pkg_cust_reg",                          # periodicity work of the failed month; rebuilt at the next start
    "_state_pkg_cust_reg/month=2025-07",           # not needed: the build resumes at 2025-09 from the 2025-08 state
    "_state_pkg_cust_reg/month=2025-09",           # left by the failed attempt; rewritten when 2025-09 is built
    "_tmp_pkg_cpty_explore",                       # October exploration scratch
    "_pkg_cpty_metrics_state/313742575a",          # counterparty metrics state of two older signatures
    "_pkg_cpty_metrics_state/b11811d8b2",          #   (current: c6405f6803, which the 2026-10 build used: kept)
    "_tmp_pkg_cpty_metrics/313742575a",            # counterparty metrics work of the same older signatures
    "_tmp_pkg_cpty_metrics/b11811d8b2",
    "_tmp_pkg_cust_metrics_v3",                    # v3 work folder: v3 is no longer built
    "pkg_cust_churn_state",                        # v2 churn state: only needed to build another v2 month
    "pkg_cust_churn_state_v3",                     # v3 churn state: only needed to build another v3 month
    "_tmp_pkg_regularity",                         # pkg_payment_regularity / forecast-check scratch (analyses done)
    "_tmp_pkg_stop_spike",                         # pkg_stop_spike_check scratch (analysis done)
    "_tmp_pkg_cpty_map_v2",                        # entity map v2 build scratch (map built)
    "_tmp_pkg_fi_canonical",                       # canonical bank build scratch (tables built)
]
DROP_TABLES = [
    # your call (not in the list by default):
    # "pkg_cust_pair_month",        # v2 pair table, 0.25 TB: does fraud or the Streamlit app read it?
    # "pkg_cust_pair_month_v3",     # v3 pair table, 0.26 TB: does pkg_attr_cpty_features read it?
    # "pkg_cust_fi_month",          # v3 customer × bank (v4 has pkg_cust_fi_month_v4)
]
DRY_RUN = True                  # True = only show what would be removed

# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)
import datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("pkg_storage_inventory")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.executor.memoryOverhead", "1g").config("spark.driver.memory", "4g")
         .config("spark.dynamicAllocation.maxExecutors", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
def strip_scheme(p):
    p = str(p)
    if "://" in p:
        p = "/" + p.split("://", 1)[1].split("/", 1)[1]
    return p.rstrip("/")
def tb(v):
    return v / 1e12
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))
def table(pdf):
    return pdf.to_html(index=False, na_rep="", float_format=lambda x: f"{x:,.3f}")

ROOT_URI = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
ROOT = strip_scheme(ROOT_URI)
FS = fs_for(ROOT_URI)
out(f"<p>Database folder: <code>{_html.escape(ROOT)}</code></p>")

In [ ]:
# ---- §2 Inventory (read-only) --------------------------------------------------------------------
cs = FS.getContentSummary(JPath(ROOT_URI))
q_space, used = cs.getSpaceQuota(), cs.getSpaceConsumed()
logical = cs.getLength()
repl = used / logical if logical else float("nan")

# tables and the folders they point to
loc2tab = {}
for r in spark.sql(f"SHOW TABLES IN {DB}").collect():
    t = r["tableName"]
    if r["isTemporary"]:
        continue
    try:
        loc = [x[1] for x in spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.{t}").collect() if x[0] == "Location"]
        if loc:
            loc2tab.setdefault(strip_scheme(loc[0]), []).append(t)
    except Exception:
        pass

# what each PKG folder is for (from the PKG notebooks); anything else: "not a PKG folder: check the owner"
KNOWN = [
    ("pkg_cust_metrics_monthly", "customer metrics v2: superseded by v3 / v4; the attrition Streamlit app still reads v2", "candidate once the app moves"),
    ("pkg_cust_pair_month", "customer pair table v2: superseded", "candidate"),
    ("pkg_cust_metrics_dictionary", "v2 dictionary (small)", "keep with v2"),
    ("pkg_cust_churn_state", "v2 churn state: only needed to build another v2 month", "candidate"),
    ("pkg_cust_metrics_monthly_v3", "customer metrics v3: attrition m4 reads it", "keep until attrition moves to v4"),
    ("pkg_cust_pair_month_v3", "customer pair table v3: same definition as v4, not read by attrition", "candidate"),
    ("pkg_cust_fi_month", "customer × bank v3 (no suffix)", "candidate (v4 has pkg_cust_fi_month_v4)"),
    ("pkg_cust_churn_state_v3", "v3 churn state: only needed to build another v3 month", "candidate"),
    ("pkg_cust_metrics_monthly_v4", "customer metrics v4.1 (in use)", "keep"),
    ("pkg_cust_pair_month_v4", "customer pair table v4 (periodicity reads it)", "keep"),
    ("pkg_cust_fi_month_v4", "customer × bank v4", "keep"),
    ("pkg_cust_churn_state_v4", "v4 churn state: the next month needs it", "keep"),
    ("_tmp_pkg_cust_metrics_v4", "customer metrics work folder (static lookups + unfinished months)", "month= folders removable when no run is going"),
    ("_tmp_pkg_cust_metrics_v3", "customer metrics v3 work folder", "candidate"),
    ("pkg_cust_reg_stream", "periodicity streams (in use)", "keep"),
    ("pkg_cust_reg_month", "periodicity month table", "keep"),
    ("pkg_cust_reg_total", "periodicity total table", "keep"),
    ("pkg_cust_reg_calib", "periodicity calibration", "keep"),
    ("_state_pkg_cust_reg", "periodicity stream state (latest 2 months; the next month needs the latest)", "keep the latest month only"),
    ("_work_pkg_cust_reg", "periodicity work folder of the month being built (rebuilt at every start)", "removable when no run is going"),
    ("_tmp_pkg_regularity", "pkg_payment_regularity scratch (stream days, 24 months)", "candidate: analysis done"),
    ("_tmp_pkg_stop_spike", "pkg_stop_spike_check scratch", "candidate: analysis done"),
    ("_tmp_pkg_reference", "pkg_reference_data scratch", "candidate when no run is going"),
    ("_staging_pkg_reference", "pkg_reference_data staging", "candidate when no run is going"),
    ("_pkg_tm_staging", "TM customer-month staging of the customer fact", "removable after the TM build finishes"),
    ("pkg_tm_cust_month", "TM customer × month (in use)", "keep"),
    ("pkg_tm_cust_prod_month", "TM customer × product × month (in use)", "keep"),
    (".Trash", "HDFS trash inside the quota: deleted files still count until it is emptied", "candidate"),
]
def known(name):
    for k, what, sug in KNOWN:
        if name == k:
            return what, sug
    return "", ""

def dir_row(st, level):
    p = strip_scheme(st.getPath().toString())
    c = FS.getContentSummary(st.getPath())
    name = p[len(ROOT) + 1:]
    base = name.split("/")[-1]
    try:
        kids = [x.getPath().getName() for x in FS.listStatus(st.getPath()) if x.isDirectory()]
    except Exception:
        kids = []
    months = sorted(k.split("=", 1)[1] for k in kids if k.startswith("month="))
    what, sug = known(base)
    return {"folder": name, "level": level, "table": ", ".join(loc2tab.get(p, [])),
            "TB consumed": tb(c.getSpaceConsumed()), "share of quota": c.getSpaceConsumed() / q_space if q_space > 0 else None,
            "GB logical": c.getLength() / 1e9, "files": c.getFileCount(),
            "months": len(months), "first": months[0] if months else "", "last": months[-1] if months else "",
            "owner": st.getOwner(), "changed": dt.datetime.fromtimestamp(st.getModificationTime() / 1000).strftime("%Y-%m-%d"),
            "what it is": what or ("" if st.getOwner() else ""), "suggestion": sug}

rows = []
for st in FS.listStatus(JPath(ROOT_URI)):
    if not st.isDirectory():
        continue
    rows.append(dir_row(st, 1))
    if st.getPath().getName().startswith(DEEP):
        for st2 in FS.listStatus(st.getPath()):
            if st2.isDirectory():
                rows.append(dir_row(st2, 2))
INV = pd.DataFrame(rows)
top = INV[INV.level == 1].sort_values("TB consumed", ascending=False)
deep = INV[INV.level == 2].sort_values("TB consumed", ascending=False).head(40)
by_owner = (top.groupby("owner", as_index=False)[["TB consumed", "GB logical"]].sum().sort_values("TB consumed", ascending=False))
cand = top[top.suggestion.str.startswith(("candidate", "removable"))]
INV.to_csv("pkg_storage_inventory.csv", index=False)

summary = pd.DataFrame([
    {"item": "space quota (TB, consumed = size × replication)", "value": f"{tb(q_space):,.2f}" if q_space > 0 else "none"},
    {"item": "space consumed (TB)", "value": f"{tb(used):,.2f}"},
    {"item": "free (TB)", "value": f"{tb(q_space - used):,.3f}" if q_space > 0 else ""},
    {"item": "logical size of the files (TB)", "value": f"{tb(logical):,.2f}"},
    {"item": "replication (consumed ÷ logical)", "value": f"{repl:.2f}"},
    {"item": "PKG candidates listed below (TB consumed)", "value": f"{cand['TB consumed'].sum():,.2f}"}])
out("<h3>§2 Storage of the database folder</h3>", table(summary),
    "<h4>By owner (top-level folders)</h4>", table(by_owner),
    "<h4>Top-level folders, largest first</h4>", table(top.drop(columns=["level"]).head(60)),
    "<h4>Inside state, work, temp and trash folders (largest 40)</h4>", table(deep.drop(columns=["level"])),
    "<p style='font-size:12px'>Full list saved to pkg_storage_inventory.csv next to the notebook. Suggestions cover PKG "
    "folders only; folders owned by others are theirs to decide.</p>")

In [ ]:
# ---- §2b Look inside the largest folders, and space held by HDFS snapshots ------------------------
# getContentSummary (and the quota) count files kept by snapshots too; listing the files only sees current files.
# A large gap between the two means deleted files are still held by a snapshot and deleting more will not free quota.
def visible_bytes(jp):
    it, n = FS.listFiles(jp, True), 0
    while it.hasNext():
        n += it.next().getLen()
    return n
look = []
for rel in LOOK_INSIDE:
    jp = JPath(ROOT_URI + "/" + rel)
    if not FS.exists(jp):
        look.append({"folder": rel, "child": "(missing)"})
        continue
    c = FS.getContentSummary(jp)
    look.append({"folder": rel, "child": "(all)", "TB consumed": tb(c.getSpaceConsumed()), "GB logical": c.getLength() / 1e9,
                 "GB in current files": visible_bytes(jp) / 1e9})
    for st in FS.listStatus(jp):
        cc = FS.getContentSummary(st.getPath())
        look.append({"folder": rel, "child": st.getPath().getName() + ("/" if st.isDirectory() else ""),
                     "TB consumed": tb(cc.getSpaceConsumed()), "GB logical": cc.getLength() / 1e9,
                     "GB in current files": visible_bytes(st.getPath()) / 1e9 if st.isDirectory() else st.getLen() / 1e9,
                     "changed": dt.datetime.fromtimestamp(st.getModificationTime() / 1000).strftime("%Y-%m-%d")})
LOOK = pd.DataFrame(look)
# snapshots on the database folder or any folder above it
snaps = []
parts = ROOT.strip("/").split("/")
for i in range(len(parts), 0, -1):
    p = "/" + "/".join(parts[:i])
    try:
        sp = JPath(ROOT_URI[:len(ROOT_URI) - len(ROOT)] + p + "/.snapshot")
        if FS.exists(sp):
            names = [x.getPath().getName() for x in FS.listStatus(sp)]
            snaps.append({"folder": p, "snapshots": len(names), "latest": sorted(names)[-1] if names else ""})
    except Exception as e:
        pass
total_vis = visible_bytes(JPath(ROOT_URI))
out("<h3>§2b Inside the largest folders</h3>", table(LOOK),
    "<h4>Snapshots</h4>",
    table(pd.DataFrame(snaps)) if snaps else "<p>No snapshot folder found on the database folder or above it (or not visible to you).</p>",
    f"<p>Whole database folder: {logical / 1e9:,.0f} GB counted by the quota (÷ replication) vs {total_vis / 1e9:,.0f} GB in "
    f"current files. A gap means snapshots hold deleted files: ask the platform team to release them.</p>")

In [ ]:
# ---- §3 Clean-up (only what is listed in DROP_TABLES / DELETE_PATHS; DRY_RUN = True lists without removing) ----
plan = []
for t in DROP_TABLES:
    loc = [strip_scheme(x[1]) for x in spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.{t}").collect() if x[0] == "Location"]
    plan.append(("table", t, loc[0] if loc else ""))
for p in DELETE_PATHS:
    full = p if p.startswith("/") else f"{ROOT}/{p.strip('/')}"
    plan.append(("folder", "", full.rstrip("/")))
bad = [x for x in plan if x[2] and not (x[2] + "/").startswith(ROOT + "/") or x[2] == ROOT]
if bad:
    raise RuntimeError(f"refusing: outside the database folder or the folder itself: {bad}")
res = []
for kind, t, p in plan:
    jp = JPath(ROOT_URI + p[len(ROOT):]) if p else None
    size = FS.getContentSummary(jp).getSpaceConsumed() if jp is not None and FS.exists(jp) else 0
    done = "would remove" if DRY_RUN else ""
    if not DRY_RUN:
        if kind == "table":
            spark.sql(f"DROP TABLE IF EXISTS {DB}.{t}")
        if jp is not None and FS.exists(jp):
            FS.delete(jp, True)          # Hadoop API delete: skips the trash, so the quota frees at once
        done = "removed"
    res.append({"kind": kind, "table": t, "folder": p, "TB consumed": tb(size), "status": done})
after = FS.getContentSummary(JPath(ROOT_URI))
out("<h3>§3 Clean-up" + (" (dry run)" if DRY_RUN else "") + "</h3>",
    table(pd.DataFrame(res)) if res else "<p>Nothing listed in DROP_TABLES or DELETE_PATHS.</p>",
    f"<p>Space consumed now {tb(after.getSpaceConsumed()):,.2f} TB of {tb(after.getSpaceQuota()):,.2f} TB.</p>"
    if after.getSpaceQuota() > 0 else "",
    "<p>After dropping tables: INVALIDATE METADATA in Impala for them.</p>" if (res and not DRY_RUN and DROP_TABLES) else "")